# RUL Prediction Baseline
Baseline regression model for Remaining Useful Life (RUL).

In [ ]:
from src.io import load_backblaze_smart
from src.cleaning import handle_missing_values, drop_constant_features, normalize_columns
from src.preprocessing import sort_by_serial_and_date, convert_date_column
from src.features import generate_time_based_features
from src.labeling import compute_rul
from src.split import split_by_serial_number
from src.models import get_baseline_regressors
from src.evaluation import regression_metrics

import numpy as np

df = load_backblaze_smart()
if df is None:
    print("Dataset missing. Please add Backblaze SMART files to DATA_DIR.")
else:
    df = handle_missing_values(df)
    df = drop_constant_features(df)
    df = normalize_columns(df)
    df = convert_date_column(df)
    df = sort_by_serial_and_date(df)
    df = generate_time_based_features(df)
    df = compute_rul(df)
    df = df[df["failure"] == 1]
    train_df, test_df = split_by_serial_number(df)
    feature_cols = [c for c in train_df.columns if c.startswith("smart_")]
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df["RUL"].fillna(0)
    X_test = test_df[feature_cols].fillna(0)
    y_test = test_df["RUL"].fillna(0)
    models = get_baseline_regressors()
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)
        print(f"Model: {name}")
        print(metrics)